# Тестовое задание на Windows через Colab

Полная инструкция: GUIDE_RU.md в обновлённом ZIP. На Windows ничего из Python/CUDA/Ubuntu для этого сценария не устанавливается. Здесь используются NVIDIA GPU Google, Qwen2.5-1.5B-Instruct и два QLoRA-адаптера.

Сначала Runtime → Change runtime type → GPU. Выполняйте ячейки по порядку кнопкой ▶. Run all не используйте: данные, оценки и отчёт нужно проверить вручную. Этот стартовый notebook не содержит заранее выполненного обучения или выдуманных результатов.

## Загрузка проекта

Выберите один исходный ZIP комплекта. Для восстановления новой машины можно выбрать последний checkpoint ZIP. После обычного перезапуска Python эту ячейку повторно не запускайте: папка остаётся.

In [1]:
from google.colab import files
from pathlib import Path
import tempfile, zipfile, shutil
project_dir = Path('/content/llm-style-task')
if project_dir.exists():
    raise RuntimeError('Папка проекта уже существует. Для продолжения выполните проверку/run ниже; не распаковывайте стартовый ZIP поверх результатов.')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert len(zip_names) == 1, 'Загрузите один ZIP проекта'
with tempfile.TemporaryDirectory(dir='/content') as temporary_dir:
    with zipfile.ZipFile(zip_names[0]) as archive:
        archive.extractall(temporary_dir)
    candidates = [p.parent for p in Path(temporary_dir).rglob('train.py') if (p.parent / 'common.py').is_file() and (p.parent / 'data/raw/test_inputs.jsonl').is_file()]
    assert len(candidates) == 1, 'ZIP должен содержать один проект с train.py, common.py и data/raw'
    shutil.copytree(candidates[0], project_dir)
print('Проект загружен:', project_dir)


Saving llm_style_step_by_step.zip to llm_style_step_by_step.zip
Проект загружен: /content/llm-style-task


## Проверка назначенной GPU

Должна появиться таблица NVIDIA GPU. Если GPU нет, сначала проверьте тип среды.

In [2]:
%cd /content/llm-style-task
!python --version
!nvidia-smi


/content/llm-style-task
Python 3.13.15
Thu Oct 15 10:31:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------

## Установка библиотек

Дождитесь окончания. Затем Runtime → Restart session / Перезапустить сеанс. Не выбирайте удаление среды. Продолжайте с проверки после перезапуска; ZIP заново не загружается.

In [3]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-train.txt'], check=True)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-r', 'requirements-train.txt'], returncode=0)

## Проверка после перезапуска и определение run

Повторяйте эту ячейку после каждого перезапуска Python. Команда run запускает скрипт и останавливается при его ошибке. T4 автоматически использует FP16; BF16=False на T4 — нормально.

In [1]:
%cd /content/llm-style-task
import subprocess, sys
from pathlib import Path
from google.colab import files
import torch, transformers, peft, bitsandbytes
def run(*arguments):
    subprocess.run([sys.executable, *arguments], check=True)
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('peft', peft.__version__)
print('bitsandbytes', bitsandbytes.__version__)
assert torch.cuda.is_available(), 'CUDA GPU не найдена: сначала включите облачную GPU'
print('GPU', torch.cuda.get_device_name(0))
print('VRAM GiB', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print('BF16 supported', torch.cuda.is_bf16_supported())


/content/llm-style-task
torch 2.8.0+cu126
transformers 4.57.1
peft 0.17.1
bitsandbytes 0.47.0
GPU Tesla T4
VRAM GiB 14.56
BF16 supported True


In [2]:
from pathlib import Path

p = Path("/content/llm-style-task/common.py")
code = p.read_text(encoding="utf-8")

code = code.replace(
    "torch.cuda.is_bf16_supported()",
    "torch.cuda.is_bf16_supported(including_emulation=False)"
)

p.write_text(code, encoding="utf-8")

run("-c", "from common import cuda_dtype; print('Тип вычислений:', cuda_dtype())")

In [3]:
import importlib
import common

importlib.reload(common)
print("Тип вычислений:", common.cuda_dtype())

Тип вычислений: torch.float16


## Подготовка данных

После команды скачайте DATA_REVIEW.md и прочитайте все 50 пар. Шесть предложенных правок targets документированы отдельно. Не добавляйте пять финальных test-текстов в обучение.

In [5]:
import subprocess
import sys

def run(*arguments):
    command = [sys.executable, "-u", *arguments]

    with subprocess.Popen(
        command,
        cwd="/content/llm-style-task",
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)

        return_code = process.wait()
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, command)

run("prepare_data.py")


{
  "seed": 42,
  "test_count": 5,
  "styles": {
    "type1": {
      "train_count": 20,
      "val_count": 5,
      "train_ids": [
        "type1-001",
        "type1-002",
        "type1-003",
        "type1-004",
        "type1-005",
        "type1-006",
        "type1-007",
        "type1-008",
        "type1-009",
        "type1-011",
        "type1-012",
        "type1-014",
        "type1-015",
        "type1-016",
        "type1-018",
        "type1-021",
        "type1-022",
        "type1-023",
        "type1-024",
        "type1-025"
      ],
      "val_ids": [
        "type1-010",
        "type1-013",
        "type1-017",
        "type1-019",
        "type1-020"
      ]
    },
    "type2": {
      "train_count": 20,
      "val_count": 5,
      "train_ids": [
        "type2-001",
        "type2-002",
        "type2-003",
        "type2-004",
        "type2-005",
        "type2-006",
        "type2-007",
        "type2-008",
        "type2-009",
        "type2-011",
        "

In [6]:
files.download('/content/llm-style-task/DATA_REVIEW.md')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()

name = "data_review_fix.zip"
assert name in uploaded, "Выберите файл data_review_fix.zip"

archive_path = Path("/content/data_review_fix.zip")
archive_path.write_bytes(uploaded[name])

script_path = Path("/content/apply_data_review.py")

with zipfile.ZipFile(archive_path) as archive:
    script_path.write_bytes(archive.read("apply_data_review.py"))

run(
    str(script_path),
    "--project", "/content/llm-style-task",
    "--patch", str(archive_path),
)

Saving data_review_fix.zip to data_review_fix.zip
{
  "seed": 42,
  "test_count": 5,
  "styles": {
    "type1": {
      "train_count": 20,
      "val_count": 5,
      "train_ids": [
        "type1-001",
        "type1-002",
        "type1-003",
        "type1-004",
        "type1-005",
        "type1-006",
        "type1-007",
        "type1-008",
        "type1-009",
        "type1-011",
        "type1-012",
        "type1-014",
        "type1-015",
        "type1-016",
        "type1-018",
        "type1-021",
        "type1-022",
        "type1-023",
        "type1-024",
        "type1-025"
      ],
      "val_ids": [
        "type1-010",
        "type1-013",
        "type1-017",
        "type1-019",
        "type1-020"
      ]
    },
    "type2": {
      "train_count": 20,
      "val_count": 5,
      "train_ids": [
        "type2-001",
        "type2-002",
        "type2-003",
        "type2-004",
        "type2-005",
        "type2-006",
        "type2-007",
        "type2-008",
 

## Наивный baseline

Запускайте после проверки данных. Первый прогон скачает модель и закрепит её ревизию. Десять validation-ответов, без адаптеров.

In [8]:
run('generate.py', '--mode', 'baseline', '--split', 'validation')


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
type1/type1-010: 214 токенов; подсказки=[]
type1/type1-013: 210 токенов; подсказки=['CHECK_NEW_NUMBERS:1']
type1/type1-017: 193 токенов; подсказки=[]
type1/type1-019: 130 токенов; подсказки=[]
type1/type1-020: 232 токенов; подсказки=['CHECK_NEW_NUMBERS:1']
type2/type2-010: 196 токенов; подсказки=['CHECK_LEADING_EMOJI']
type2/type2-013: 160 токенов; подсказки=['CHECK_LEADING_EMOJI']
type2/type2-017: 133 токенов; подсказки=['CHECK_PARAGRAPH_BREAK', 'CHECK_LEADING_EMOJI']
type2/type2-019: 164 токенов; подсказки=['CHECK_LEADING_EMOJI']
type2/type2-020: 174 токенов; подсказки=['CHECK_LEADING_EMOJI']
Сохранено: /content/llm-style-task/artifacts/baseline_validation.jsonl


## Подробный промпт без адаптера

Контроль эффекта промпта на тех же validation-исходниках.

In [9]:
run('generate.py', '--mode', 'prompt-only', '--split', 'validation')


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
type1/type1-010: 133 токенов; подсказки=[]
type1/type1-013: 117 токенов; подсказки=[]
type1/type1-017: 82 токенов; подсказки=[]
type1/type1-019: 37 токенов; подсказки=[]
type1/type1-020: 92 токенов; подсказки=[]
type2/type2-010: 145 токенов; подсказки=['CHECK_PARAGRAPH_BREAK', 'CHECK_LEADING_EMOJI']
type2/type2-013: 112 токенов; подсказки=['CHECK_PARAGRAPH_BREAK', 'CHECK_LEADING_EMOJI']
type2/type2-017: 140 токенов; подсказки=['CHECK_PARAGRAPH_BREAK', 'CHECK_LEADING_EMOJI']
type2/type2-019: 133 токенов; подсказки=['CHECK_LEADING_EMOJI']
type2/type2-020: 76 токенов; подсказки=['CHECK_PARAGRAPH_BREAK', 'CHECK_LEADING_EMOJI']
Сохранено: /content/llm-style-task/artifacts/prompt-only_validation.jsonl


## Обучение адаптера type1

Дождитесь сохранения adapters/type1. Не запускайте type2 параллельно.

In [10]:
run('train.py', '--style', 'type1')


`torch_dtype` is deprecated! Use `dtype` instead!
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.

  3%|▎         | 1/30 [00:02<01:13,  2.53s/it]
                                              
{'loss': 1.0329, 'grad_norm': 2.0521225929260254, 'learning_rate': 0.0, 'epoch': 0.1}

  7%|▋         | 2/30 [00:04<01:00,  2.16s/it]
                                              
{'loss': 1.1885, 'grad_norm': 1.6767876148223877, 'learning_rate': 3.3333333333333335e-05, 'epoch': 0.2}

 10%|█         | 3/30 [00:06<00:56,  2.09s/it]
                                              
{'loss': 1.1031, 'grad_norm': 1.5865286588668823, 'learning_rate': 6.666666666666667e-05, 'epoch': 0.3}

 13%|█▎        | 4/

## Резервная копия после type1

Скачайте ZIP на Windows и дождитесь полного скачивания. Это промежуточный проект, а не финальная сдача.

In [11]:
run('package_submission.py', '--output', '/content/checkpoint_type1.zip')
files.download('/content/checkpoint_type1.zip')


/content/checkpoint_type1.zip: 46 файлов


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Обучение адаптера type2

Второй адаптер обучается независимо, поверх исходной базы.

In [12]:
run('train.py', '--style', 'type2')


`torch_dtype` is deprecated! Use `dtype` instead!
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.

  3%|▎         | 1/30 [00:02<01:14,  2.55s/it]
                                              
{'loss': 1.0263, 'grad_norm': 1.7405117750167847, 'learning_rate': 0.0, 'epoch': 0.1}

  7%|▋         | 2/30 [00:04<00:56,  2.03s/it]
                                              
{'loss': 1.0144, 'grad_norm': 1.8836232423782349, 'learning_rate': 3.3333333333333335e-05, 'epoch': 0.2}

 10%|█         | 3/30 [00:06<00:53,  1.98s/it]
                                              
{'loss': 1.1673, 'grad_norm': 1.9279017448425293, 'learning_rate': 6.666666666666667e-05, 'epoch': 0.3}

 13%|█▎        | 4/

## Резервная копия обоих адаптеров

In [13]:
run('package_submission.py', '--output', '/content/checkpoint_both_adapters.zip')
files.download('/content/checkpoint_both_adapters.zip')


/content/checkpoint_both_adapters.zip: 58 файлов


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Проверка adapted на validation

Будет создан REVIEW.md с тридцатью ответами трёх режимов. Следующий этап — ручная оценка на Windows.

In [14]:
run('generate.py', '--mode', 'adapted', '--split', 'validation')
run('make_review.py')


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
type1/type1-010: 112 токенов; подсказки=[]
type1/type1-013: 110 токенов; подсказки=[]
type1/type1-017: 124 токенов; подсказки=[]
type1/type1-019: 104 токенов; подсказки=[]
type1/type1-020: 106 токенов; подсказки=[]
type2/type2-010: 150 токенов; подсказки=[]
type2/type2-013: 114 токенов; подсказки=[]
type2/type2-017: 124 токенов; подсказки=[]
type2/type2-019: 118 токенов; подсказки=[]
type2/type2-020: 75 токенов; подсказки=[]
Сохранено: /content/llm-style-task/artifacts/adapted_validation.jsonl
Создан REVIEW.md: оценки не заполнены автоматически.


## Скачивание REVIEW.md

Откройте файл в VS Code/Блокноте Windows. Заполните четыре оценки в каждой из 30 строк: факты 0/1, стиль/структура/язык 1–5. Не меняйте первые три столбца, сохраняйте имя REVIEW.md.

In [15]:
files.download('/content/llm-style-task/REVIEW.md')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
from google.colab import files
from pathlib import Path
import re

uploaded = files.upload()
assert "REVIEW.md" in uploaded, "Выберите файл REVIEW.md"

root = Path("/content/llm-style-task")
(root / "REVIEW.md").write_bytes(uploaded["REVIEW.md"])

p = root / "review_scores.py"
code = re.sub(
    r"Оценки поставлены[^.]*\.",
    "Редакторские оценки даны относительно предоставленных исходников.",
    p.read_text(encoding="utf-8")
)
code = code.replace(
    "# Сводная ручная оценка",
    "# Сводная редакторская оценка"
)
p.write_text(code, encoding="utf-8")

run("review_scores.py")

Saving REVIEW.md to REVIEW (1).md


AssertionError: Выберите файл REVIEW.md

In [17]:
from pathlib import Path
import re

assert len(uploaded) == 1, "Должен быть загружен один файл"

filename, content = next(iter(uploaded.items()))
root = Path("/content/llm-style-task")

(root / "REVIEW.md").write_bytes(content)
print(f"Загруженный файл {filename} сохранён как REVIEW.md")

p = root / "review_scores.py"
code = re.sub(
    r"Оценки поставлены[^.]*\.",
    "Редакторские оценки даны относительно предоставленных исходников.",
    p.read_text(encoding="utf-8")
)
code = code.replace(
    "# Сводная ручная оценка",
    "# Сводная редакторская оценка"
)
p.write_text(code, encoding="utf-8")

run("review_scores.py")

Загруженный файл REVIEW (1).md сохранён как REVIEW.md
# Сводная редакторская оценка

Редакторские оценки даны относительно предоставленных исходников. Скрипт только проверяет заполнение и считает сводные значения. Выборка — 10 validation-случаев на режим.

| Режим | Ответов | Существенных искажений | Средний стиль | Средняя структура | Средний язык |
|---|---:|---:|---:|---:|---:|
| baseline | 10 | 4 | 1.3 | 1.6 | 3.3 |
| prompt-only | 10 | 4 | 2.2 | 2.9 | 4.1 |
| adapted | 10 | 4 | 2.5 | 4.3 | 3.4 |



## Загрузка заполненной оценки

Выберите исправленный REVIEW.md с Windows. Затем посчитайте сводку следующей ячейкой.

In [ ]:
edited = files.upload()
assert 'REVIEW.md' in edited, 'Выберите файл с точным именем REVIEW.md'
Path('/content/llm-style-task/REVIEW.md').write_bytes(edited['REVIEW.md'])
print('REVIEW.md обновлён')


In [ ]:
run('review_scores.py')


## Финальные генерации

Выполняйте после выбора настройки по validation. Пять test-источников → два стиля → десять постов.

In [18]:
run('generate.py', '--mode', 'adapted', '--split', 'test')


`torch_dtype` is deprecated! Use `dtype` instead!
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
type1/1: 174 токенов; подсказки=[]
type1/2: 185 токенов; подсказки=[]
type1/3: 146 токенов; подсказки=[]
type1/4: 246 токенов; подсказки=[]
type1/5: 206 токенов; подсказки=[]
type2/1: 150 токенов; подсказки=['CHECK_LEADING_EMOJI']
type2/2: 258 токенов; подсказки=[]
type2/3: 128 токенов; подсказки=[]
type2/4: 219 токенов; подсказки=[]
type2/5: 325 токенов; подсказки=[]
Сохранено: /content/llm-style-task/artifacts/adapted_test.jsonl


In [1]:
from pathlib import Path

root = Path("/content/llm-style-task")

required = [
    "make_results.py",
    "common.py",
    "PRODUCTION.md",
    "serve.sh",
    "REVIEW.md",
    "REVIEW_SUMMARY.md",
    "data/data_report.json",
    "artifacts/train_type1.json",
    "artifacts/train_type2.json",
    "artifacts/baseline_validation.jsonl",
    "artifacts/prompt-only_validation.jsonl",
    "artifacts/adapted_validation.jsonl",
    "artifacts/adapted_test.jsonl",
    "adapters/type1/adapter_model.safetensors",
    "adapters/type2/adapter_model.safetensors",
    "outputs_type1.txt",
    "outputs_type2.txt",
]

missing = [
    name for name in required
    if not (root / name).is_file()
]

if missing:
    print("НЕ НАЙДЕНЫ ФАЙЛЫ:")
    for name in missing:
        print("-", name)
    print("\nСначала нужно восстановить проект из резервного архива.")
else:
    print("ВСЕ НУЖНЫЕ ФАЙЛЫ СОХРАНИЛИСЬ.")
    print("Можно создавать RESULTS.md.")

НЕ НАЙДЕНЫ ФАЙЛЫ:
- make_results.py
- common.py
- PRODUCTION.md
- serve.sh
- REVIEW.md
- REVIEW_SUMMARY.md
- data/data_report.json
- artifacts/train_type1.json
- artifacts/train_type2.json
- artifacts/baseline_validation.jsonl
- artifacts/prompt-only_validation.jsonl
- artifacts/adapted_validation.jsonl
- artifacts/adapted_test.jsonl
- adapters/type1/adapter_model.safetensors
- adapters/type2/adapter_model.safetensors
- outputs_type1.txt
- outputs_type2.txt

Сначала нужно восстановить проект из резервного архива.


In [2]:
from google.colab import files
from pathlib import Path
import io
import shutil
import tempfile
import zipfile

project = Path("/content/llm-style-task")

assert not project.exists(), (
    "Папка проекта уже существует. Ничего не перезаписываем: "
    "пришлите это сообщение для проверки."
)

uploaded = files.upload()

assert len(uploaded) == 1, "Выберите один резервный ZIP"

archive_name, archive_bytes = next(iter(uploaded.items()))

with tempfile.TemporaryDirectory(dir="/content") as temporary:
    temporary = Path(temporary)

    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        assert all(
            (temporary / name).resolve().is_relative_to(
                temporary.resolve()
            )
            for name in archive.namelist()
        ), "Некорректные пути в архиве"

        archive.extractall(temporary)

    candidates = [
        p.parent
        for p in temporary.rglob("train.py")
        if (p.parent / "common.py").is_file()
        and (p.parent / "data/raw/test_inputs.jsonl").is_file()
    ]

    assert len(candidates) == 1, (
        "Не удалось однозначно найти проект в архиве"
    )

    shutil.copytree(candidates[0], project)

print("ПРОЕКТ ВОССТАНОВЛЕН:", project)
print("\nПроверка сохранённых результатов:")

important = [
    "adapters/type1/adapter_model.safetensors",
    "adapters/type2/adapter_model.safetensors",
    "artifacts/train_type1.json",
    "artifacts/train_type2.json",
    "artifacts/baseline_validation.jsonl",
    "artifacts/prompt-only_validation.jsonl",
    "artifacts/adapted_validation.jsonl",
    "artifacts/adapted_test.jsonl",
    "REVIEW.md",
    "REVIEW_SUMMARY.md",
    "outputs_type1.txt",
    "outputs_type2.txt",
]

for name in important:
    status = "ЕСТЬ" if (project / name).is_file() else "НЕТ"
    print(f"{status}: {name}")

Saving checkpoint_both_adapters.zip to checkpoint_both_adapters.zip
ПРОЕКТ ВОССТАНОВЛЕН: /content/llm-style-task

Проверка сохранённых результатов:
ЕСТЬ: adapters/type1/adapter_model.safetensors
ЕСТЬ: adapters/type2/adapter_model.safetensors
ЕСТЬ: artifacts/train_type1.json
ЕСТЬ: artifacts/train_type2.json
ЕСТЬ: artifacts/baseline_validation.jsonl
ЕСТЬ: artifacts/prompt-only_validation.jsonl
НЕТ: artifacts/adapted_validation.jsonl
НЕТ: artifacts/adapted_test.jsonl
НЕТ: REVIEW.md
НЕТ: REVIEW_SUMMARY.md
НЕТ: outputs_type1.txt
НЕТ: outputs_type2.txt


In [3]:
from google.colab import files
from pathlib import Path
import io
import subprocess
import sys
import tempfile
import zipfile

root = Path("/content/llm-style-task")

assert (root / "make_results.py").is_file(), (
    "Сначала нужно восстановить основной проект из checkpoint."
)

uploaded_restore = files.upload()

assert len(uploaded_restore) == 1, (
    "Выберите один файл restore_saved_results.zip"
)

archive_name, archive_bytes = next(
    iter(uploaded_restore.items())
)

with tempfile.TemporaryDirectory(dir="/content") as temporary:
    temporary = Path(temporary)

    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
        assert all(
            (temporary / name).resolve().is_relative_to(
                temporary.resolve()
            )
            for name in archive.namelist()
        ), "Некорректные пути в архиве"

        archive.extractall(temporary)

    script = temporary / "restore_saved_results.py"
    inputs = temporary / "saved_files"

    assert script.is_file(), (
        "Выбран другой архив. Нужен restore_saved_results.zip"
    )

    result = subprocess.run(
        [
            sys.executable,
            "-u",
            str(script),
            "--root",
            str(root),
            "--inputs",
            str(inputs),
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print(result.stdout)
    result.check_returncode()

Saving restore_saved_results.zip to restore_saved_results.zip
# Сводная редакторская оценка

Редакторские оценки даны относительно предоставленных исходников. Скрипт только проверяет заполнение и считает сводные значения. Выборка — 10 validation-случаев на режим.

| Режим | Ответов | Существенных искажений | Средний стиль | Средняя структура | Средний язык |
|---|---:|---:|---:|---:|---:|
| baseline | 10 | 4 | 1.3 | 1.6 | 3.3 |
| prompt-only | 10 | 4 | 2.2 | 2.9 | 4.1 |
| adapted | 10 | 4 | 2.5 | 4.3 | 3.4 |


ВОССТАНОВЛЕНИЕ ЗАВЕРШЕНО. Нового обучения и генерации не было.
Созданы восстановленные записи и RECOVERY.md; исходные веса и логи checkpoint сохранены.
Следующий шаг: make_results.py, если RESULTS.md ещё не создан.



In [5]:
import subprocess
import sys
from pathlib import Path

root = Path("/content/llm-style-task")

assert not (root / "RESULTS.md").exists(), (
    "RESULTS.md уже существует. Сначала скачайте его, "
    "чтобы не перезаписать внесённые изменения."
)

result = subprocess.run(
    [sys.executable, "-u", "make_results.py"],
    cwd=root,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding="utf-8",
    errors="replace",
)

print(result.stdout)
result.check_returncode()

AssertionError: RESULTS.md уже существует. Сначала скачайте его, чтобы не перезаписать внесённые изменения.

In [6]:
from google.colab import files

files.download("/content/llm-style-task/RESULTS.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
from pathlib import Path

root = Path("/content/llm-style-task")

required = [
    "common.py",
    "finalize_submission.py",
    "package_submission.py",
    "review_scores.py",
    "REVIEW.md",
    "RECOVERY.md",
    "artifacts/model_lock.json",
    "artifacts/adapted_test.jsonl",
    "outputs_type1.txt",
    "outputs_type2.txt",
]

for style in ("type1", "type2"):
    required.extend([
        f"artifacts/train_{style}.json",
        f"adapters/{style}/adapter_model.safetensors",
        f"adapters/{style}/adapter_config.json",
    ])

for mode in ("baseline", "prompt-only", "adapted"):
    required.append(
        f"artifacts/{mode}_validation.jsonl"
    )

missing = [
    name for name in required
    if not (root / name).is_file()
]

if missing:
    print("НЕ НАЙДЕНЫ:")
    for name in missing:
        print("-", name)

assert not missing, (
    "Сначала нужно восстановить отсутствующие файлы. "
    "Упаковку пока не запускайте."
)

print("Проект доступен. Можно загружать финальный отчёт.")

НЕ НАЙДЕНЫ:
- common.py
- finalize_submission.py
- package_submission.py
- review_scores.py
- REVIEW.md
- RECOVERY.md
- artifacts/model_lock.json
- artifacts/adapted_test.jsonl
- outputs_type1.txt
- outputs_type2.txt
- artifacts/train_type1.json
- adapters/type1/adapter_model.safetensors
- adapters/type1/adapter_config.json
- artifacts/train_type2.json
- adapters/type2/adapter_model.safetensors
- adapters/type2/adapter_config.json
- artifacts/baseline_validation.jsonl
- artifacts/prompt-only_validation.jsonl
- artifacts/adapted_validation.jsonl


AssertionError: Сначала нужно восстановить отсутствующие файлы. Упаковку пока не запускайте.

In [2]:
from google.colab import files
from pathlib import Path
import io
import shutil
import subprocess
import sys
import tempfile
import zipfile

project = Path("/content/llm-style-task")

assert not project.exists() or not any(project.iterdir()), (
    "Папка проекта уже содержит файлы. "
    "Ничего не перезаписываем — пришлите это сообщение."
)

print(
    "Выберите одновременно два архива: "
    "checkpoint_both_adapters.zip и restore_saved_results.zip"
)

uploaded = files.upload()

assert len(uploaded) == 2, (
    "Нужно выбрать ровно два указанных ZIP-архива."
)

def find_archive(prefix):
    matches = [
        content
        for name, content in uploaded.items()
        if name.lower().startswith(prefix)
        and name.lower().endswith(".zip")
    ]

    assert len(matches) == 1, (
        f"Не найден один архив с началом имени: {prefix}"
    )

    return matches[0]

def extract_safely(content, destination):
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(io.BytesIO(content)) as archive:
        assert all(
            (destination / name).resolve().is_relative_to(
                destination.resolve()
            )
            for name in archive.namelist()
        ), "Некорректные пути внутри ZIP"

        archive.extractall(destination)

checkpoint_bytes = find_archive("checkpoint_both_adapters")
recovery_bytes = find_archive("restore_saved_results")

with tempfile.TemporaryDirectory(dir="/content") as temporary:
    temporary = Path(temporary)

    checkpoint_dir = temporary / "checkpoint"
    recovery_dir = temporary / "recovery"

    extract_safely(checkpoint_bytes, checkpoint_dir)
    extract_safely(recovery_bytes, recovery_dir)

    candidates = [
        p.parent
        for p in checkpoint_dir.rglob("train.py")
        if (p.parent / "common.py").is_file()
        and (p.parent / "data/raw/test_inputs.jsonl").is_file()
    ]

    assert len(candidates) == 1, (
        "Не удалось однозначно найти проект в checkpoint."
    )

    shutil.copytree(
        candidates[0],
        project,
        dirs_exist_ok=True
    )

    print("Основной проект восстановлен:", project)

    script = recovery_dir / "restore_saved_results.py"

    assert script.is_file(), (
        "В маленьком архиве не найден скрипт восстановления."
    )

    result = subprocess.run(
        [
            sys.executable,
            "-u",
            str(script),
            "--root",
            str(project),
            "--inputs",
            str(recovery_dir / "saved_files"),
        ],
        cwd=project,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    print(result.stdout)
    result.check_returncode()

print(
    "\nГОТОВО: проект и сохранённые результаты восстановлены."
)
print(
    "Теперь выполните пункт 3 проверки файлов, "
    "затем пункт 4 загрузки финального RESULTS.md."
)

Выберите одновременно два архива: checkpoint_both_adapters.zip и restore_saved_results.zip


Saving checkpoint_both_adapters.zip to checkpoint_both_adapters.zip
Saving restore_saved_results.zip to restore_saved_results.zip
Основной проект восстановлен: /content/llm-style-task
# Сводная редакторская оценка

Редакторские оценки даны относительно предоставленных исходников. Скрипт только проверяет заполнение и считает сводные значения. Выборка — 10 validation-случаев на режим.

| Режим | Ответов | Существенных искажений | Средний стиль | Средняя структура | Средний язык |
|---|---:|---:|---:|---:|---:|
| baseline | 10 | 4 | 1.3 | 1.6 | 3.3 |
| prompt-only | 10 | 4 | 2.2 | 2.9 | 4.1 |
| adapted | 10 | 4 | 2.5 | 4.3 | 3.4 |


ВОССТАНОВЛЕНИЕ ЗАВЕРШЕНО. Нового обучения и генерации не было.
Созданы восстановленные записи и RECOVERY.md; исходные веса и логи checkpoint сохранены.
Следующий шаг: make_results.py, если RESULTS.md ещё не создан.


ГОТОВО: проект и сохранённые результаты восстановлены.
Теперь выполните пункт 3 проверки файлов, затем пункт 4 загрузки финального RESULTS.md.


In [3]:
from pathlib import Path

root = Path("/content/llm-style-task")

required = [
    "common.py",
    "finalize_submission.py",
    "package_submission.py",
    "review_scores.py",
    "REVIEW.md",
    "RECOVERY.md",
    "artifacts/model_lock.json",
    "artifacts/adapted_test.jsonl",
    "outputs_type1.txt",
    "outputs_type2.txt",
]

for style in ("type1", "type2"):
    required.extend([
        f"artifacts/train_{style}.json",
        f"adapters/{style}/adapter_model.safetensors",
        f"adapters/{style}/adapter_config.json",
    ])

for mode in ("baseline", "prompt-only", "adapted"):
    required.append(
        f"artifacts/{mode}_validation.jsonl"
    )

missing = [
    name for name in required
    if not (root / name).is_file()
]

if missing:
    print("НЕ НАЙДЕНЫ:")
    for name in missing:
        print("-", name)

assert not missing, (
    "Сначала нужно восстановить отсутствующие файлы. "
    "Упаковку пока не запускайте."
)

print("Проект доступен. Можно загружать финальный отчёт.")

Проект доступен. Можно загружать финальный отчёт.


In [4]:
from google.colab import files
from pathlib import Path

root = Path("/content/llm-style-task")

uploaded_report = files.upload()

assert len(uploaded_report) == 1, (
    "Выберите один финальный RESULTS.md"
)

report_bytes = next(iter(uploaded_report.values()))
report_text = report_bytes.decode("utf-8-sig")

assert "[ЗАПОЛНИТЬ" not in report_text, (
    "Выбран незаполненный черновик."
)

assert "Шкалы оценки:" in report_text, (
    "Выбрана предыдущая версия отчёта."
)

assert "Первые приоритеты:" in report_text, (
    "Выбрана предыдущая версия отчёта."
)

(root / "RESULTS.md").write_text(
    report_text,
    encoding="utf-8"
)

print("Финальный RESULTS.md загружен в проект.")
print("Размер:", (root / "RESULTS.md").stat().st_size, "байт")

Saving RESULTS.md to RESULTS.md
Финальный RESULTS.md загружен в проект.
Размер: 52948 байт


## Скачивание и проверка десяти постов

Сверьте все посты с таблицей фактов в GUIDE_RU.md. Запустите две ячейки скачивания по очереди.

In [19]:
files.download('/content/llm-style-task/outputs_type1.txt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
files.download('/content/llm-style-task/outputs_type2.txt')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Создание отчёта и фиксация окружения

Отчёт использует настоящие ответы и вашу сводку оценок. Далее заполните человеческие выводы на Windows. После ручного заполнения make_results.py повторно не запускается.

In [ ]:
run('make_results.py')
with open('/content/llm-style-task/requirements-train-lock.txt', 'w', encoding='utf-8') as output:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=output, check=True)


In [ ]:
files.download('/content/llm-style-task/RESULTS.md')


## Загрузка заполненного отчёта

На Windows замените все [ЗАПОЛНИТЬ] реальными наблюдениями, временем и статусом серверного теста. Сохраните имя RESULTS.md и выберите его здесь. vLLM в этом notebook не устанавливается; серверный кейс описан в отчёте как план, если своего NVIDIA-сервера нет.

In [ ]:
edited = files.upload()
assert 'RESULTS.md' in edited, 'Выберите файл с точным именем RESULTS.md'
Path('/content/llm-style-task/RESULTS.md').write_bytes(edited['RESULTS.md'])
print('RESULTS.md обновлён')


## Итоговая проверка и упаковка

Создаются полный ZIP с адаптерами и небольшой ZIP для GitHub через браузер. Скрипт проверяет заполнение и структуру, а не смысловую верность постов.

In [ ]:
run('finalize_submission.py', '--output-dir', '/content')


## Скачать полный проект

Сохраните архив на Windows. Он содержит бинарные веса адаптеров.

In [ ]:
files.download('/content/submission.zip')


## Скачать файлы для GitHub

Распакуйте github_upload.zip на Windows и загрузите его содержимое в корень репозитория через сайт GitHub. Большие бинарные веса исключены и остаются в полном архиве.

In [ ]:
files.download('/content/github_upload.zip')


## Сохранить выполненный notebook

File → Download → Download .ipynb. Добавьте как executed_colab.ipynb в браузерный экспорт перед загрузкой на GitHub. Порядок публикации — в GUIDE_RU.md. Сдача: ссылка на репозиторий, outputs_type1.txt, outputs_type2.txt и RESULTS.md.